# Source Code: Implementation Modules

All core implementation code for TypiClust reproduction, AL baselines, and evaluation frameworks.

## SimCLR Encoder & Embedding Extraction


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np


class ResNet18Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.resnet = models.resnet18(weights=None)
        self.resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.resnet.maxpool = nn.Identity()
        self.resnet.fc = nn.Identity()

    def forward(self, x):
        return self.resnet(x)


def load_encoder(weights_path, device='cuda'):
    encoder = ResNet18Encoder()
    encoder.load_state_dict(torch.load(weights_path, map_location=device))
    encoder = encoder.to(device)
    encoder.eval()
    return encoder


def extract_embeddings(encoder, dataloader, device='cuda'):
    encoder.eval()
    all_embeddings = []
    all_labels = []
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            h = encoder(images)
            h = torch.nn.functional.normalize(h, dim=1)  # L2 normalize (paper: F.1)
            all_embeddings.append(h.cpu().numpy())
            all_labels.append(labels.numpy())
    return np.vstack(all_embeddings), np.hstack(all_labels)


## Typicality Computation


In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors


def compute_typicality(embeddings, k=20):
    nn = NearestNeighbors(n_neighbors=k + 1)
    nn.fit(embeddings)
    distances, _ = nn.kneighbors(embeddings)
    distances = distances[:, 1:]  # exclude self
    return np.mean(distances, axis=1) ** -1


## K-Means Clustering


In [ ]:
import numpy as np
from sklearn.cluster import KMeans


def cluster_standard(embeddings, budget, random_state=42, **kwargs):
    kmeans = KMeans(n_clusters=budget, random_state=random_state, n_init=10)
    kmeans.fit(embeddings)
    return kmeans.labels_


def cluster_overclustering(embeddings, budget, cluster_mult=5, random_state=42, **kwargs):
    n_clusters = budget * cluster_mult
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    kmeans.fit(embeddings)

    points_per_cluster = np.bincount(kmeans.labels_)
    sorted_clusters = np.argsort(points_per_cluster)[-budget:]
    new_labels = np.full_like(kmeans.labels_, -1)

    for i, cluster in enumerate(sorted_clusters):
        new_labels[kmeans.labels_ == cluster] = i

    return new_labels

## PCA & UMAP Preprocessing


In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import umap

def preprocess_identity(embeddings, **kwargs):
    return embeddings, None


def preprocess_pca(embeddings, n_components=100, fitted_model=None, **kwargs):
    if fitted_model is None:
        pca = PCA(n_components=n_components)
        transformed = pca.fit_transform(embeddings)
        return transformed, pca
    else:
        return fitted_model.transform(embeddings), fitted_model

def preprocess_umap(embeddings, n_components=2, fitted_model=None, random_state=42, **kwargs):
    if fitted_model is None:
        reducer = umap.UMAP(n_components=n_components, random_state=random_state)
        transformed = reducer.fit_transform(embeddings)
        return transformed, reducer
    else:
        return fitted_model.transform(embeddings), fitted_model

## Selection Strategies (All 9: Random, Uncertainty, Margin, Entropy, CoreSet, BADGE, BALD, DBAL, TypiClust)


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression


def select_random(embeddings, budget, rng=None, **kwargs):
    """Random selection baseline."""
    if rng is None:
        rng = np.random.RandomState(42)
    return rng.choice(len(embeddings), size=budget, replace=False)


def select_uncertainty(embeddings, labels, labelled_idx, budget, **kwargs):
    """Select points with lowest max softmax probability (most uncertain)."""
    unlabelled_idx = np.setdiff1d(np.arange(len(embeddings)), labelled_idx)
    if len(labelled_idx) == 0:
        return np.random.RandomState(42).choice(len(embeddings), size=budget, replace=False)
    clf = LogisticRegression(max_iter=1000, C=100)
    clf.fit(embeddings[labelled_idx], labels[labelled_idx])
    probs = clf.predict_proba(embeddings[unlabelled_idx])
    max_probs = probs.max(axis=1)
    selected = unlabelled_idx[np.argsort(max_probs)[:budget]]
    return selected


def select_margin(embeddings, labels, labelled_idx, budget, **kwargs):
    """Select points with smallest margin between top-2 softmax probabilities."""
    unlabelled_idx = np.setdiff1d(np.arange(len(embeddings)), labelled_idx)
    if len(labelled_idx) == 0:
        return np.random.RandomState(42).choice(len(embeddings), size=budget, replace=False)
    clf = LogisticRegression(max_iter=1000, C=100)
    clf.fit(embeddings[labelled_idx], labels[labelled_idx])
    probs = clf.predict_proba(embeddings[unlabelled_idx])
    sorted_probs = np.sort(probs, axis=1)
    margins = sorted_probs[:, -1] - sorted_probs[:, -2]
    selected = unlabelled_idx[np.argsort(margins)[:budget]]
    return selected


def select_entropy(embeddings, labels, labelled_idx, budget, **kwargs):
    """Select points with highest softmax entropy."""
    unlabelled_idx = np.setdiff1d(np.arange(len(embeddings)), labelled_idx)
    if len(labelled_idx) == 0:
        return np.random.RandomState(42).choice(len(embeddings), size=budget, replace=False)
    clf = LogisticRegression(max_iter=1000, C=100)
    clf.fit(embeddings[labelled_idx], labels[labelled_idx])
    probs = clf.predict_proba(embeddings[unlabelled_idx])
    entropy = -np.sum(probs * np.log(probs + 1e-10), axis=1)
    selected = unlabelled_idx[np.argsort(entropy)[-budget:]]
    return selected


def select_coreset(embeddings, labelled_idx, budget, **kwargs):
    """Greedy k-center coreset selection — maximize min distance to labelled set.
    Uses incremental distance updates to avoid recomputing all pairwise distances."""
    unlabelled_idx = np.setdiff1d(np.arange(len(embeddings)), labelled_idx)
    if len(labelled_idx) == 0:
        seed = np.random.RandomState(42).choice(len(embeddings))
        selected = [seed]
        unlabelled_mask = np.ones(len(embeddings), dtype=bool)
        unlabelled_mask[seed] = False
        # Initialize min distances to the seed
        min_dists = np.linalg.norm(embeddings - embeddings[seed], axis=1)
        min_dists[seed] = -1
        remaining = budget - 1
    else:
        selected = list(labelled_idx)
        unlabelled_mask = np.ones(len(embeddings), dtype=bool)
        unlabelled_mask[labelled_idx] = False
        # Initialize min distances to all labelled points
        min_dists = np.full(len(embeddings), np.inf)
        for idx in labelled_idx:
            d = np.linalg.norm(embeddings - embeddings[idx], axis=1)
            min_dists = np.minimum(min_dists, d)
        min_dists[labelled_idx] = -1
        remaining = budget

    for _ in range(remaining):
        # Select the unlabelled point with largest min distance
        candidates = np.where(unlabelled_mask)[0]
        best_pos = candidates[np.argmax(min_dists[candidates])]
        selected.append(best_pos)
        unlabelled_mask[best_pos] = False
        # Update min distances with the newly selected point
        new_dists = np.linalg.norm(embeddings - embeddings[best_pos], axis=1)
        min_dists = np.minimum(min_dists, new_dists)
        min_dists[best_pos] = -1

    if len(labelled_idx) == 0:
        return np.array(selected)
    return np.array(selected[len(labelled_idx):])


def _mc_dropout_predict(embeddings, labels, labelled_idx, unlabelled_idx,
                        n_forward=20, n_classes=10, hidden=128, n_epochs=100, dropout=0.5):
    """Train a small network with dropout on labelled data,
    then run MC dropout inference on unlabelled data."""
    import torch
    import torch.nn as nn

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    d = embeddings.shape[1]

    # Small MLP with dropout
    model = nn.Sequential(
        nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(hidden, n_classes)
    ).to(device)

    # Train
    X_train = torch.FloatTensor(embeddings[labelled_idx]).to(device)
    y_train = torch.LongTensor(labels[labelled_idx]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    model.train()
    for _ in range(n_epochs):
        optimizer.zero_grad()
        loss = criterion(model(X_train), y_train)
        loss.backward()
        optimizer.step()

    # MC dropout inference (keep dropout ON)
    model.train()  # keeps dropout active
    X_test = torch.FloatTensor(embeddings[unlabelled_idx]).to(device)
    all_probs = []
    with torch.no_grad():
        for _ in range(n_forward):
            logits = model(X_test)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)

    return np.stack(all_probs)  # (n_forward, n_unlabelled, n_classes)


def select_bald(embeddings, labels, labelled_idx, budget, **kwargs):
    """BALD: Bayesian Active Learning by Disagreement.
    Select points with highest mutual information between predictions and model parameters.
    MI = H[y|x] - E[H[y|x,w]] (entropy of mean - mean of entropies)."""
    unlabelled_idx = np.setdiff1d(np.arange(len(embeddings)), labelled_idx)
    if len(labelled_idx) == 0:
        return np.random.RandomState(42).choice(len(embeddings), size=budget, replace=False)

    all_probs = _mc_dropout_predict(embeddings, labels, labelled_idx, unlabelled_idx)
    # Mean prediction across forward passes
    mean_probs = all_probs.mean(axis=0)
    # Entropy of mean prediction
    H_mean = -np.sum(mean_probs * np.log(mean_probs + 1e-10), axis=1)
    # Mean entropy across forward passes
    entropies = -np.sum(all_probs * np.log(all_probs + 1e-10), axis=2)
    mean_H = entropies.mean(axis=0)
    # BALD score = mutual information
    bald_scores = H_mean - mean_H
    selected = unlabelled_idx[np.argsort(bald_scores)[-budget:]]
    return selected


def select_dbal(embeddings, labels, labelled_idx, budget, **kwargs):
    """DBAL: Deep Bayesian Active Learning.
    Select points with highest average predictive entropy under MC dropout."""
    unlabelled_idx = np.setdiff1d(np.arange(len(embeddings)), labelled_idx)
    if len(labelled_idx) == 0:
        return np.random.RandomState(42).choice(len(embeddings), size=budget, replace=False)

    all_probs = _mc_dropout_predict(embeddings, labels, labelled_idx, unlabelled_idx)
    # Average entropy across MC samples
    entropies = -np.sum(all_probs * np.log(all_probs + 1e-10), axis=2)
    mean_entropy = entropies.mean(axis=0)
    selected = unlabelled_idx[np.argsort(mean_entropy)[-budget:]]
    return selected


def select_badge(embeddings, labels, labelled_idx, budget, **kwargs):
    """BADGE: Batch Active learning by Diverse Gradient Embeddings.
    Uses gradient embeddings from the last layer of a trained linear model,
    then applies k-means++ initialization to select diverse uncertain points."""
    unlabelled_idx = np.setdiff1d(np.arange(len(embeddings)), labelled_idx)
    if len(labelled_idx) == 0:
        return np.random.RandomState(42).choice(len(embeddings), size=budget, replace=False)

    clf = LogisticRegression(max_iter=1000, C=100)
    clf.fit(embeddings[labelled_idx], labels[labelled_idx])

    # Compute gradient embeddings for unlabelled points
    probs = clf.predict_proba(embeddings[unlabelled_idx])
    predicted = probs.argmax(axis=1)
    n_classes = probs.shape[1]

    # Gradient embedding: for each point, the gradient of the loss w.r.t.
    # the last layer weights is (p - y) ⊗ x, where y is one-hot predicted label
    # We use the predicted label's gradient: (1 - p_hat) * x for the predicted class
    grad_embeddings = []
    for i in range(len(unlabelled_idx)):
        p = probs[i]
        x = embeddings[unlabelled_idx[i]]
        # Gradient for predicted class
        one_hot = np.zeros(n_classes)
        one_hot[predicted[i]] = 1.0
        grad = np.outer(p - one_hot, x).flatten()
        grad_embeddings.append(grad)
    grad_embeddings = np.array(grad_embeddings)

    # k-means++ initialization on gradient embeddings to select diverse uncertain points
    selected_local = []
    # First point: highest gradient norm (most uncertain)
    norms = np.linalg.norm(grad_embeddings, axis=1)
    first = np.argmax(norms)
    selected_local.append(first)

    for _ in range(budget - 1):
        # Distance to nearest selected point
        dists = np.linalg.norm(
            grad_embeddings[:, None] - grad_embeddings[selected_local][None, :],
            axis=2
        )
        min_dists = dists.min(axis=1)
        min_dists[selected_local] = 0
        # Proportional to distance squared
        probs_select = min_dists ** 2
        probs_select /= probs_select.sum() + 1e-10
        next_idx = np.random.RandomState(42).choice(len(grad_embeddings), p=probs_select)
        selected_local.append(next_idx)

    return unlabelled_idx[np.array(selected_local)]


def select_max_typicality(embeddings, typicality, cluster_labels, budget, **kwargs):
    selected = []
    for cluster in range(budget):
        cluster_idx = np.where(cluster_labels == cluster)[0]
        best = cluster_idx[np.argmax(typicality[cluster_idx])]
        selected.append(best)
    return np.array(selected)


def select_hybrid(embeddings, typicality, cluster_labels, budget, alpha=0.5, **kwargs):

    selected = []

    cluster_order = []
    for c in range(budget):
        cluster_idx = np.where(cluster_labels == c)[0]
        cluster_order.append((c, np.max(typicality[cluster_idx])))
    cluster_order.sort(key=lambda x: x[1], reverse=True)

    first_cluster = cluster_order[0][0]
    cluster_idx = np.where(cluster_labels == first_cluster)[0]
    best = cluster_idx[np.argmax(typicality[cluster_idx])]
    selected.append(best)

    for cluster in cluster_order[1:]:
        cluster_idx = np.where(cluster_labels == cluster[0])[0]
        candidate_embeddings = embeddings[cluster_idx]
        dist = np.linalg.norm(candidate_embeddings[:, None] - embeddings[selected], axis=2)
        min_distance = dist.min(axis=1)

        norm_distance = (min_distance - min_distance.min()) / (min_distance.max() - min_distance.min() + 1e-8)
        norm_typicality = (typicality[cluster_idx] - typicality[cluster_idx].min()) / (typicality[cluster_idx].max() - typicality[cluster_idx].min() + 1e-8)

        score = alpha * norm_typicality + (1 - alpha) * norm_distance

        best = cluster_idx[np.argmax(score)]
        selected.append(best)

    return np.array(selected)


## Evaluation Frameworks (Fully Supervised, SS Embedding, Semi-Supervised) & Multi-Round AL Loop


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors

import torchvision.models as models

from src.selection import (
    select_random, select_uncertainty, select_margin,
    select_entropy, select_coreset, select_badge,
    select_bald, select_dbal, select_max_typicality
)


# ============================================================
# Framework 1: Fully Supervised (train ResNet-18 from scratch)
# ============================================================

def _make_resnet18_cifar(num_classes=10):
    """ResNet-18 adapted for CIFAR-10 (32x32 images)."""
    model = models.resnet18(weights=None, num_classes=num_classes)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model


def train_supervised(train_images, train_labels, test_images, test_labels,
                     n_epochs=200, lr=0.025, device='cuda'):
    """Train ResNet-18 from scratch on labelled images and evaluate.
    Follows paper's F.2.1: SGD with 0.9 momentum (Nesterov), cosine LR,
    random crops + horizontal flips, weight re-init each call."""
    device = torch.device(device if torch.cuda.is_available() else 'cpu')
    # Re-initialize weights each time (paper: prevents overconfident predictions)
    model = _make_resnet18_cifar().to(device)
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    criterion = nn.CrossEntropyLoss()

    # Data augmentation: random crops + horizontal flips (as per paper)
    train_tensor = torch.FloatTensor(train_images)
    train_labels_tensor = torch.LongTensor(train_labels)
    train_ds = TensorDataset(train_tensor, train_labels_tensor)
    train_loader = DataLoader(train_ds, batch_size=min(len(train_images), 64),
                              shuffle=True, drop_last=False)

    # Training with per-image augmentation
    model.train()
    for epoch in range(n_epochs):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            # Per-image random horizontal flip
            flip_mask = torch.rand(x.size(0)) > 0.5
            x[flip_mask] = torch.flip(x[flip_mask], dims=[3])
            # Per-image random crop (pad 4, crop 32)
            x = nn.functional.pad(x, (4, 4, 4, 4), mode='reflect')
            augmented = torch.zeros(x.size(0), x.size(1), 32, 32, device=device)
            for img_i in range(x.size(0)):
                ci = torch.randint(0, 8, (1,)).item()
                cj = torch.randint(0, 8, (1,)).item()
                augmented[img_i] = x[img_i, :, ci:ci+32, cj:cj+32]
            optimizer.zero_grad()
            loss = criterion(model(augmented), y)
            loss.backward()
            optimizer.step()
        scheduler.step()

    # Evaluation
    model.eval()
    test_ds = TensorDataset(
        torch.FloatTensor(test_images), torch.LongTensor(test_labels))
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=1)
            correct += (preds == y).sum().item()
            total += len(y)

    return correct / total


# ============================================================
# Framework 2: SS Embedding (linear probe on frozen embeddings)
# ============================================================

def linear_probe(train_embeddings, train_labels, test_embeddings, test_labels):
    clf = LogisticRegression(max_iter=1000, C=100)
    clf.fit(train_embeddings, train_labels)
    preds = clf.predict(test_embeddings)
    return accuracy_score(test_labels, preds)


# ============================================================
# Framework 3: Semi-Supervised (Pseudo-labelling on embeddings)
# ============================================================
# Paper uses FlexMatch (WideResNet-28, 400k iterations) which is
# computationally infeasible. We approximate with iterative
# pseudo-labelling on frozen SimCLR embeddings with per-class
# adaptive thresholds inspired by FlexMatch's curriculum.

def semi_supervised_eval(labelled_idx, labels,
                         train_images, test_images, test_labels,
                         embeddings=None, test_embeddings=None,
                         n_iterations=5, initial_threshold=0.95,
                         max_pseudo_per_iter=5000):
    """
    Semi-supervised evaluation using iterative pseudo-labelling
    on frozen SimCLR embeddings with per-class adaptive thresholds.
    """
    train_emb = embeddings[labelled_idx]
    train_lab = labels[labelled_idx].copy()
    labelled_emb = train_emb.copy()
    labelled_lab = train_lab.copy()
    n_classes = 10

    unlabelled_mask = np.ones(len(embeddings), dtype=bool)
    unlabelled_mask[labelled_idx] = False

    class_thresholds = np.full(n_classes, initial_threshold)

    for iteration in range(n_iterations):
        clf = LogisticRegression(max_iter=500, C=100)
        clf.fit(labelled_emb, labelled_lab)

        unlabelled_idx = np.where(unlabelled_mask)[0]
        if len(unlabelled_idx) == 0:
            break

        probs = clf.predict_proba(embeddings[unlabelled_idx])
        max_probs = probs.max(axis=1)
        predicted = probs.argmax(axis=1)

        confident = np.zeros(len(unlabelled_idx), dtype=bool)
        for c in range(n_classes):
            class_mask = predicted == c
            if class_mask.sum() > 0:
                confident[class_mask] = max_probs[class_mask] >= class_thresholds[c]

        if confident.sum() == 0:
            class_thresholds *= 0.95
            continue

        confident_positions = np.where(confident)[0]
        if len(confident_positions) > max_pseudo_per_iter:
            top = np.argsort(max_probs[confident_positions])[-max_pseudo_per_iter:]
            confident_positions = confident_positions[top]

        pseudo_idx = unlabelled_idx[confident_positions]
        pseudo_labels = predicted[confident_positions]

        labelled_emb = np.vstack([labelled_emb, embeddings[pseudo_idx]])
        labelled_lab = np.concatenate([labelled_lab, pseudo_labels])
        unlabelled_mask[pseudo_idx] = False

        class_counts = np.bincount(labelled_lab.astype(int), minlength=n_classes)
        max_count = class_counts.max()
        for c in range(n_classes):
            ratio = class_counts[c] / (max_count + 1e-8)
            class_thresholds[c] = initial_threshold * ratio + (1 - ratio) * 0.5

    clf = LogisticRegression(max_iter=1000, C=100)
    clf.fit(labelled_emb, labelled_lab)
    preds = clf.predict(test_embeddings)
    return accuracy_score(test_labels, preds)


# ============================================================
# Evaluation helpers
# ============================================================

def evaluate_selection(selected_indices, embeddings, labels, test_embeddings, test_labels):
    sel_emb = embeddings[selected_indices]
    sel_lab = labels[selected_indices]
    accuracy = linear_probe(sel_emb, sel_lab, test_embeddings, test_labels)
    n_classes = len(np.unique(sel_lab))
    class_counts = np.bincount(sel_lab, minlength=10)
    tv_distance = 0.5 * np.sum(np.abs(class_counts / len(sel_lab) - 1.0 / 10))
    return {
        'accuracy': accuracy,
        'n_classes_covered': n_classes,
        'tv_distance': tv_distance,
        'selected_labels': sel_lab.tolist(),
    }


def random_baseline(embeddings, labels, test_embeddings, test_labels, budget, n_seeds=30):
    accs = []
    for seed in range(n_seeds):
        rng = np.random.RandomState(seed)
        idx = rng.choice(len(embeddings), size=budget, replace=False)
        acc = linear_probe(embeddings[idx], labels[idx], test_embeddings, test_labels)
        accs.append(acc)
    return {'mean': np.mean(accs), 'std': np.std(accs), 'all': accs}


def _compute_typicality(embeddings, k=20):
    """Compute typicality as inverse mean distance to k nearest neighbours."""
    nn_model = NearestNeighbors(n_neighbors=k + 1)
    nn_model.fit(embeddings)
    distances, _ = nn_model.kneighbors(embeddings)
    distances = distances[:, 1:]
    return np.mean(distances, axis=1) ** -1


# ============================================================
# Multi-round AL loop (supports all 3 frameworks)
# ============================================================

def run_al_rounds(embeddings, labels, test_embeddings, test_labels,
                  strategy, budget_per_round=10, n_rounds=5, n_reps=10,
                  random_state=42, framework='ss_embedding',
                  train_images=None, test_images=None,
                  all_embeddings=None, all_labels=None):
    """
    Run multi-round active learning experiment.

    Args:
        strategy: 'random', 'uncertainty', 'margin', 'entropy', 'coreset', 'typiclust'
        framework: 'fully_supervised', 'ss_embedding', 'semi_supervised'
        train_images: raw images (needed for fully_supervised)
        test_images: raw test images (needed for fully_supervised)
        all_embeddings: full training pool embeddings (needed for semi_supervised)
        all_labels: full training pool labels (needed for semi_supervised)
    """
    budgets = [(i + 1) * budget_per_round for i in range(n_rounds)]
    all_accs = np.zeros((n_reps, n_rounds))

    for rep in range(n_reps):
        seed = random_state + rep
        rng = np.random.RandomState(seed)
        labelled_idx = np.array([], dtype=int)

        for round_i in range(n_rounds):
            # --- Selection ---
            if strategy == 'random':
                unlabelled = np.setdiff1d(np.arange(len(embeddings)), labelled_idx)
                new_idx = rng.choice(unlabelled, size=budget_per_round, replace=False)

            elif strategy == 'uncertainty':
                new_idx = select_uncertainty(
                    embeddings, labels, labelled_idx, budget_per_round)

            elif strategy == 'margin':
                new_idx = select_margin(
                    embeddings, labels, labelled_idx, budget_per_round)

            elif strategy == 'entropy':
                new_idx = select_entropy(
                    embeddings, labels, labelled_idx, budget_per_round)

            elif strategy == 'coreset':
                new_idx = select_coreset(
                    embeddings, labelled_idx, budget_per_round)

            elif strategy == 'badge':
                new_idx = select_badge(
                    embeddings, labels, labelled_idx, budget_per_round)

            elif strategy == 'bald':
                new_idx = select_bald(
                    embeddings, labels, labelled_idx, budget_per_round)

            elif strategy == 'dbal':
                new_idx = select_dbal(
                    embeddings, labels, labelled_idx, budget_per_round)

            elif strategy == 'typiclust':
                # Paper: K = min(|L_{i-1}| + B, max_clusters)
                # max_clusters = 500 for CIFAR-10
                max_clusters = 500
                unlabelled_idx = np.setdiff1d(
                    np.arange(len(embeddings)), labelled_idx)
                unlabelled_emb = embeddings[unlabelled_idx]

                n_clusters = min(len(labelled_idx) + budget_per_round, max_clusters)
                kmeans = KMeans(
                    n_clusters=n_clusters, random_state=seed, n_init=10)
                kmeans.fit(unlabelled_emb)

                # Find uncovered clusters
                all_cluster_labels = kmeans.predict(embeddings)
                covered_clusters = set()
                if len(labelled_idx) > 0:
                    covered_clusters = set(all_cluster_labels[labelled_idx])

                uncovered = [c for c in range(n_clusters)
                             if c not in covered_clusters]

                # Paper: drop clusters with < 5 samples, use min(20, cluster_size) for KNN
                new_idx_local = []
                for c in uncovered:
                    if len(new_idx_local) >= budget_per_round:
                        break
                    mask = kmeans.labels_ == c
                    cluster_idx = np.where(mask)[0]
                    cluster_size = len(cluster_idx)
                    if cluster_size < 5:
                        continue  # Paper: skip small clusters
                    # Paper: use min(20, cluster_size) neighbours
                    k_typ = min(20, cluster_size - 1)
                    if k_typ < 1:
                        continue
                    typicality = _compute_typicality(
                        unlabelled_emb[cluster_idx], k=k_typ)
                    best = cluster_idx[np.argmax(typicality)]
                    new_idx_local.append(best)

                # If not enough, fill from remaining uncovered (relaxed size filter)
                if len(new_idx_local) < budget_per_round:
                    for c in uncovered:
                        if len(new_idx_local) >= budget_per_round:
                            break
                        mask = kmeans.labels_ == c
                        cluster_idx = np.where(mask)[0]
                        if any(cluster_idx[0] == idx for idx in new_idx_local):
                            continue
                        # Pick most central point by distance to centroid
                        centroid = unlabelled_emb[cluster_idx].mean(axis=0)
                        dists = np.linalg.norm(unlabelled_emb[cluster_idx] - centroid, axis=1)
                        best = cluster_idx[np.argmin(dists)]
                        new_idx_local.append(best)

                new_idx = unlabelled_idx[new_idx_local[:budget_per_round]]

            else:
                raise ValueError(f"Unknown strategy: {strategy}")

            labelled_idx = np.concatenate([labelled_idx, new_idx]).astype(int)

            # --- Evaluation (framework-dependent) ---
            if framework == 'ss_embedding':
                acc = linear_probe(
                    embeddings[labelled_idx], labels[labelled_idx],
                    test_embeddings, test_labels)

            elif framework == 'fully_supervised':
                if train_images is None or test_images is None:
                    raise ValueError("fully_supervised requires train_images and test_images")
                acc = train_supervised(
                    train_images[labelled_idx], labels[labelled_idx],
                    test_images, test_labels)

            elif framework == 'semi_supervised':
                acc = semi_supervised_eval(
                    labelled_idx, labels,
                    train_images=None, test_images=None, test_labels=test_labels,
                    embeddings=embeddings, test_embeddings=test_embeddings)

            else:
                raise ValueError(f"Unknown framework: {framework}")

            all_accs[rep, round_i] = acc

    return {
        'budgets': budgets,
        'mean': all_accs.mean(axis=0),
        'std': all_accs.std(axis=0),
        'all': all_accs,
    }


## Pipeline Runner


In [ ]:
import numpy as np
from src.preprocessing import preprocess_identity, preprocess_pca, preprocess_umap
from src.typicality import compute_typicality
from src.clustering import cluster_overclustering, cluster_standard
from src.selection import select_max_typicality, select_hybrid
from src.evaluation import evaluate_selection

# Registry of available components
PREPROCESSORS = {
    'none': preprocess_identity,
    'pca': preprocess_pca,
    'umap': preprocess_umap,
}

CLUSTERERS = {
    'standard': cluster_standard,
    'overclustering': cluster_overclustering,
}

SELECTORS = {
    'max_typicality': select_max_typicality,
    'select_hybrid': select_hybrid,
}


def run_pipeline(embeddings, labels, test_embeddings, test_labels, config):
    """
    Run one full TypiClust pipeline with given config.

    config keys:
        preprocess: str - 'none', 'pca', 'umap'
        cluster: str - 'standard', 'overclustering'
        selection: str - 'max_typicality', 'hybrid'
        budget: int
        k_typicality: int (default 20)
        random_state: int (default 42)
        + any method-specific params (pca_dims, cluster_mult, alpha, etc.)
    """
    budget = config.get('budget', 10)
    k_typ = config.get('k_typicality', 20)
    rs = config.get('random_state', 42)

    # 1. Preprocess
    preprocess_fn = PREPROCESSORS[config.get('preprocess', 'none')]

    proc_embeddings, model = preprocess_fn(embeddings, **config)
    proc_test, _ = preprocess_fn(test_embeddings, fitted_model=model)
    # 2. Compute typicality on preprocessed embeddings
    typicality = compute_typicality(proc_embeddings, k=k_typ)

    # 3. Cluster
    cluster_fn = CLUSTERERS[config.get('cluster', 'standard')]

    
    cluster_labels = cluster_fn(proc_embeddings, budget=budget, random_state=rs, **{
        k: v for k, v in config.items() if k not in ('budget', 'random_state', 'preprocess', 'cluster', 'selection', 'k_typicality')
    })


    # 4. Select
    select_fn = SELECTORS[config.get('selection', 'max_typicality')]
    selected_indices = select_fn(
        proc_embeddings, typicality, cluster_labels, **config
    )

    # 5. Evaluate (use preprocessed embeddings for consistency)
    results = evaluate_selection(
        selected_indices, proc_embeddings, labels, proc_test, test_labels
    )
    results['config'] = config.copy()
    results['selected_indices'] = selected_indices.tolist()

    return results


def run_experiment_grid(embeddings, labels, test_embeddings, test_labels, configs):
    """Run pipeline for each config in the list. Returns list of result dicts."""
    results = []
    for i, config in enumerate(configs):
        print(f"[{i+1}/{len(configs)}] {config}")
        try:
            result = run_pipeline(embeddings, labels, test_embeddings, test_labels, config)
            results.append(result)
        except Exception as e:
            print(f"  FAILED: {e}")
            results.append({'config': config, 'error': str(e)})
    return results
